# Project Summary (Data + Features)

This notebook summarizes what has been built so far:
- Raw weather data coverage
- Daily energy aggregation
- Wastage target creation
- Feature engineering output
- Final dataset readiness for ML

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

print('✓ Libraries loaded')

✓ Libraries loaded


## 1. Daily Energy Dataset

In [4]:
daily_path = Path('../data/processed/daily_energy.csv')
daily = pd.read_csv(daily_path)
daily['date'] = pd.to_datetime(daily['date'])

print(f'Path: {daily_path}')
print(f'Shape: {daily.shape}')
print(f'Date range: {daily["date"].min()} to {daily["date"].max()}')
print(f'Households: {daily["household"].nunique()}')
print('\nHead:')
print(daily.head())

Path: ..\data\processed\daily_energy.csv
Shape: (6412, 4)
Date range: 2014-12-11 00:00:00 to 2019-05-01 00:00:00
Households: 4

Head:
        date  daily_pv_kwh  daily_load_kwh     household
0 2014-12-11           0.0             0.0  residential1
1 2014-12-12           0.0             0.0  residential1
2 2014-12-13           0.0             0.0  residential1
3 2014-12-14           0.0             0.0  residential1
4 2014-12-15           0.0             0.0  residential1


## 2. Weather Data (Combined)

In [5]:
weather_path = Path('../data/processed/weather_all_areas.csv')
weather = pd.read_csv(weather_path)
weather['date'] = pd.to_datetime(weather['date'])

print(f'Path: {weather_path}')
print(f'Shape: {weather.shape}')
print(f'Date range: {weather["date"].min()} to {weather["date"].max()}')
print(f'Areas: {weather["area"].unique()}')
print('\nHead:')
print(weather.head())

print('\nWeather summary (by area):')
print(weather.groupby('area')[['irradiance', 'temp']].describe().round(2))


Path: ..\data\processed\weather_all_areas.csv
Shape: (6570, 4)
Date range: 2014-01-02 00:00:00 to 2019-12-31 00:00:00
Areas: ['Colombo' 'Galle' 'Matara']

Head:
        date  irradiance   temp     area
0 2014-01-02      5.5836  24.98  Colombo
1 2014-01-03      5.1614  24.91  Colombo
2 2014-01-04      5.2354  24.89  Colombo
3 2014-01-05      5.9659  24.42  Colombo
4 2014-01-06      5.4946  24.77  Colombo

Weather summary (by area):
        irradiance                                              temp         \
             count  mean   std   min   25%   50%   75%   max   count   mean   
area                                                                          
Colombo     2190.0  5.63  1.22  0.54  5.05  5.93  6.47  7.65  2190.0  26.52   
Galle       2190.0  5.07  1.07  0.81  4.49  5.20  5.79  7.62  2190.0  27.49   
Matara      2190.0  5.63  1.29  0.59  5.06  5.98  6.52  7.61  2190.0  27.09   

                                                  
          std    min    25%    50%    7

## 3. Feature-Engineered Dataset

In [6]:
features_path = Path('../data/processed/features_engineered.csv')
features = pd.read_csv(features_path)
features['date'] = pd.to_datetime(features['date'])

print(f'Path: {features_path}')
print(f'Shape: {features.shape}')
print(f'Date range: {features["date"].min()} to {features["date"].max()}')
print(f'Columns: {len(features.columns)}')
print('\nHead:')
print(features.head())


Path: ..\data\processed\features_engineered.csv
Shape: (6412, 43)
Date range: 2014-12-11 00:00:00 to 2019-05-01 00:00:00
Columns: 43

Head:
        date  daily_pv_kwh  daily_load_kwh     household  net_export  \
0 2014-12-11           0.0             0.0  residential1         0.0   
1 2014-12-12           0.0             0.0  residential1         0.0   
2 2014-12-13           0.0             0.0  residential1         0.0   
3 2014-12-14           0.0             0.0  residential1         0.0   
4 2014-12-15           0.0             0.0  residential1         0.0   

   wasted_energy_kwh  export_limit_kwh  year  month  day  quarter  \
0                0.0        198071.485  2014     12   11        4   
1                0.0        198071.485  2014     12   12        4   
2                0.0        198071.485  2014     12   13        4   
3                0.0        198071.485  2014     12   14        4   
4                0.0        198071.485  2014     12   15        4   

   day_of_we

## 4. Target (Wastage) Summary

In [7]:
target = 'wasted_energy_kwh'
print(features[target].describe())
print(f'Days with wastage > 0: {(features[target] > 0).sum()}')
print(f'Total wasted energy: {features[target].sum():.2f} kWh')

count     6412.000000
mean       171.446201
std       1405.894458
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      23435.077800
Name: wasted_energy_kwh, dtype: float64
Days with wastage > 0: 154
Total wasted energy: 1099313.04 kWh


## 5. Feature Inventory

In [8]:
# Show feature groups
cols = features.columns.tolist()
time_features = [c for c in cols if c in ['year','month','day','quarter','day_of_week','day_of_year','week_of_year','is_weekend','season']]
lag_features = [c for c in cols if 'lag_' in c]
rolling_features = [c for c in cols if 'rolling_' in c]
weather_features = [c for c in cols if c in ['irradiance','temp']]
id_features = [c for c in cols if c in ['date','household','district']]
energy_features = [c for c in cols if c in ['daily_pv_kwh','daily_load_kwh','net_export','export_limit_kwh']]

print(f'Time features ({len(time_features)}): {time_features}')
print(f'Lag features ({len(lag_features)}): {lag_features[:10]} ...')
print(f'Rolling features ({len(rolling_features)}): {rolling_features[:10]} ...')
print(f'Weather features ({len(weather_features)}): {weather_features}')
print(f'Energy features ({len(energy_features)}): {energy_features}')
print(f'ID features ({len(id_features)}): {id_features}')

Time features (9): ['year', 'month', 'day', 'quarter', 'day_of_week', 'day_of_year', 'week_of_year', 'is_weekend', 'season']
Lag features (12): ['pv_lag_1d', 'load_lag_1d', 'net_export_lag_1d', 'wastage_lag_1d', 'pv_lag_3d', 'load_lag_3d', 'net_export_lag_3d', 'wastage_lag_3d', 'pv_lag_7d', 'load_lag_7d'] ...
Rolling features (12): ['pv_rolling_mean_7d', 'load_rolling_mean_7d', 'pv_rolling_std_7d', 'wastage_rolling_mean_7d', 'pv_rolling_mean_14d', 'load_rolling_mean_14d', 'pv_rolling_std_14d', 'wastage_rolling_mean_14d', 'pv_rolling_mean_30d', 'load_rolling_mean_30d'] ...
Weather features (2): ['irradiance', 'temp']
Energy features (4): ['daily_pv_kwh', 'daily_load_kwh', 'net_export', 'export_limit_kwh']
ID features (3): ['date', 'household', 'district']


## 6. Data Quality Checks

In [9]:
missing = features.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('Missing values (if any):')
print(missing if len(missing) > 0 else 'None')

print('\nDuplicate rows:', features.duplicated().sum())

print('\nHouseholds count:')
print(features['household'].value_counts())


Missing values (if any):
None

Duplicate rows: 0

Households count:
household
residential1    1603
residential3    1603
residential4    1603
residential6    1603
Name: count, dtype: int64


## 7. Ready for Model Training

If everything looks correct above, proceed to the model training notebook: 03_model_training.ipynb